## Upload tables

In [ ]:
import pandas as pd
from src import utils as src_utils
from src import PROJECT_DIR, logging

from discovery_utils.utils import (
    google,
    google_slides,
)

from discovery_utils.getters import (
    crunchbase
)

PROJECT_NAME = "2025_02_MS_asf"
OUTPUT_DIR = PROJECT_DIR / f"data/{PROJECT_NAME}/mission_radar"

In [ ]:
sheet_id = "1JzD3VqnxcsbLiFCt2YaMetvZG97paHNOPRBd8ykInNI"

## Crunchbase

In [ ]:
cols_funding_rounds = [
    "funding_round_name", 
    "org_name",
    "theme",
    "cb_url",
    "country_code", 
    "region_nesta",
    "region",
    "city",
    "year",
    "announced_on",
    "investment_type", 
    "investment_stage",
    "raised_amount_gbp",
    "raised_amount_usd",
    "raised_amount", 
    "raised_amount_currency_code",
    "post_money_valuation_usd", 
    "post_money_valuation",
    "post_money_valuation_currency_code", 
    "investor_name",
]

cols_companies = [
    "name", 
    "short_description", 
    "founded_on", 
    'created_at',    
    "cb_url", 
    "homepage_url", 
    "theme",    
    'mission_labels',
    'topic_labels',
    "rank", 
    "country_code", 
    "region_nesta",
    "region", 
    "city", 
    "status", 
    "category_list", 
    "closed_on", 
    "employee_count", 
    "email", 
    "phone", 
    "facebook_url", 
    "linkedin_url", 
    "twitter_url", 
    "logo_url", 
    "num_exits", 
    "num_funding_rounds", 
    "last_funding_on", 
    "investment_funding_gbp", 
    "num_investment_rounds", 
    "grant_funding_gbp", 
    "num_grants", 
    "total_funding_gbp", 
    "smart_money", 
]

In [ ]:
all_orgs = (
    pd.read_csv(OUTPUT_DIR / "cb_all_orgs.csv")
    .assign(region_nesta = lambda df: df.country_code.map(crunchbase.country_to_region()))
)[cols_companies]

all_funding_rounds_df = (
    pd.read_csv(OUTPUT_DIR / "cb_all_funding_rounds.csv")
    .assign(investment_stage = lambda df: df.investment_type.map(crunchbase.investment_type_to_stage()))
    .assign(region_nesta = lambda df: df.country_code.map(crunchbase.country_to_region()))
)[cols_funding_rounds]

all_ipos = pd.read_csv(OUTPUT_DIR / "cb_all_ipos.csv")
all_acquisitions = pd.read_csv(OUTPUT_DIR / "cb_all_acquisitions.csv")

In [ ]:
google.upload_data_to_gsheet(sheet_id, 
    {
        "crunchbase_funding": all_funding_rounds_df,
        "crunchbase_companies": all_orgs,
        "crunchbase_ipos": all_ipos,
        "crunchbase_aquisitions": all_acquisitions

})
google.format_gsheet(sheet_id, "crunchbase_funding", freeze_cols=2)
google.format_gsheet(sheet_id, "crunchbase_companies", freeze_cols=4)
google.format_gsheet(sheet_id, "crunchbase_ipos", freeze_cols=0)
google.format_gsheet(sheet_id, "crunchbase_aquisitions", freeze_cols=0)

In [ ]:
df = pd.read_csv(OUTPUT_DIR / "cb_growth_magnitude.csv")[["theme", "variable", "magnitude", "growth"]]
google.upload_data_to_gsheet(sheet_id, {"crunchbase_stats": df})
google.format_gsheet(sheet_id, "crunchbase_stats", freeze_cols=2)


In [ ]:
df = pd.read_csv(OUTPUT_DIR / "cb_growth_magnitude_quarterly.csv")[["theme", "variable", "magnitude", "previous_four_quarters", "growth"]]
google.upload_data_to_gsheet(sheet_id, {"crunchbase_stats_quarterly": df})
google.format_gsheet(sheet_id, "crunchbase_stats_quarterly", freeze_cols=2)

## GtR

In [ ]:
enrichment_df = (
    pd.read_csv(src_utils.OUTPUT_DIR / "gtr_labelled_projects.csv")
    .assign(topic_labels = lambda df: df.topic_labels.apply(lambda x: x.split(",")))
    .assign(mission_labels = lambda df: df.mission_labels.apply(lambda x: x.split(",")))
)

In [ ]:
cols_projects = [
    'title',
    # 'is_relevant',   
    'theme',    
    'mission_labels',
    'topic_labels',
    'status', 
    'grantCategory',
    'leadFunder',
    'abstractText',
    'techAbstractText',
    'potentialImpact',
    'start',
    'end',
    'amount',
    'url',
]

In [ ]:
df = pd.read_csv(OUTPUT_DIR / "gtr_all_projects_df.csv").merge(enrichment_df, on='id', how='left')[cols_projects]
google.upload_data_to_gsheet(sheet_id, {"ukri_projects": df})
google.format_gsheet(sheet_id, "ukri_projects", freeze_cols=2)


In [ ]:
df = pd.read_csv(OUTPUT_DIR / "gtr_all_growth_magnitude_df.csv")[["theme", "variable", "magnitude", "growth"]]
google.upload_data_to_gsheet(sheet_id, {"ukri_stats": df})
google.format_gsheet(sheet_id, "ukri_stats", freeze_cols=2)


In [ ]:
df = pd.read_csv(OUTPUT_DIR / "gtr_all_growth_magnitude_quarterly_df.csv")[["theme", "variable", "magnitude", "previous_four_quarters", "growth"]]
google.upload_data_to_gsheet(sheet_id, {"ukri_stats_quarterly": df})
google.format_gsheet(sheet_id, "ukri_stats_quarterly", freeze_cols=2)

In [ ]:
## Hansard

In [ ]:
all_growth_magnitude_df = pd.read_csv(OUTPUT_DIR / "hansard_all_growth_magnitude_df.csv")[["theme", "variable", "magnitude", "growth"]]
all_growth_magnitude_quarterly_df = pd.read_csv(OUTPUT_DIR / "hansard_all_growth_magnitude_quarterly_df.csv")[["theme", "variable", "magnitude", "previous_four_quarters", "growth"]]
all_speeches_df = pd.read_csv(OUTPUT_DIR / "hansard_all_speeches_df.csv")

In [ ]:
google.upload_data_to_gsheet(sheet_id, 
    {
        "hansard_stats": all_growth_magnitude_df,
        "hansard_stats_quarterly": all_growth_magnitude_quarterly_df,
        "hansard_speeches": all_speeches_df
})
google.format_gsheet(sheet_id, "hansard_stats", freeze_cols=2)
google.format_gsheet(sheet_id, "hansard_stats_quarterly", freeze_cols=2)
google.format_gsheet(sheet_id, "hansard_speeches", freeze_cols=2)